# AgriVision Phase 4 — fourteen-class Colab GPU training
This notebook trains real MobileNetV3-Small weights using pinned original PlantVillage RGB files. It does not modify the working tomato model or Flutter application. No evaluation results are pre-filled.

1. Upload this notebook to Google Colab.
2. Select **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU).
3. Upload `agrivision_phase4_colab_source.zip` when prompted below.
4. Run all cells in order. Keep the downloaded result archive and return it to the AgriVision project.

The upstream HF card declares CC BY-SA 3.0, but the author's loader marks that license as an assumption. Original image redistribution rights remain unresolved. This package contains metadata/code only; do not publish dataset photos or result archives containing parity photos without resolving rights. Preserve attribution to Mohanty, Hughes and Salathé (2016), DOI 10.3389/fpls.2016.01419.


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.8.0', 'torchvision==0.23.0', 'Pillow==11.3.0', 'numpy', 'scikit-learn==1.7.1', 'matplotlib==3.10.5'], check=True)
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime, then restart and run all cells.'
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.get_device_name())


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
archive_name = 'agrivision_phase4_colab_source.zip'
assert archive_name in uploaded, 'Upload the Phase 4 source ZIP, not the old tomato package.'
workspace = Path('/content/agrivision_phase4')
assert not workspace.exists(), 'Use a fresh runtime to avoid mixing previous source packages.'
workspace.mkdir()
with zipfile.ZipFile(archive_name) as archive:
    for member in archive.infolist():
        target = (workspace / member.filename).resolve()
        assert workspace.resolve() in target.parents, 'Unsafe archive path'
        assert not (member.external_attr >> 16) & 0o170000 == 0o120000, 'Symlink rejected'
    archive.extractall(workspace)
os.chdir(workspace)
sys.path.insert(0, str(workspace))
from training.multicrop.pipeline import CONFIG, CLASSES, inventory
print('Pinned source:', CONFIG['source_commit'])
print('Original RGB candidates:', len(inventory()['images']))
for c in CLASSES:
    print(c['index'], c['label'], c['expected_images'])


## Acquire original RGB files and split physical leaves
Downloads only the selected fourteen RGB directories (~253 MB image bytes); no augmented, grayscale or segmented variants. Git blob hashes verify every original file. Download is resumable within this runtime.

Exact decoded duplicates are removed. Official leaf IDs, filename-derived repeated-leaf identities and exact pixel matches are grouped before splitting. Missing official IDs are reported; collection relationships and unknown near-duplicates cannot be guaranteed from available metadata. Do not claim field generalization. Split ratios are approximately 70/15/15 because groups stay indivisible. Augmentation happens only in the training loader.


In [ ]:
subprocess.run([sys.executable, '-m', 'training.multicrop.pipeline', 'download'], check=True)
subprocess.run([sys.executable, '-m', 'training.multicrop.pipeline', 'prepare'], check=True)


In [ ]:
import json
report = json.loads(Path('artifacts/phase4_prepared/dataset_report.json').read_text())
print('Retained:', report['retained_images'], 'Exact duplicates removed:', len(report['removed_exact_duplicates']))
print('Official leaf matches:', report['official_leaf_metadata_matches'], 'Filename fallbacks:', report['filename_leaf_fallbacks'])
for label, counts in report['images_per_class'].items():
    print(label, counts, 'independent groups:', report['leaf_groups_per_class'][label])
print('Potato Healthy:', report['images_per_class']['Potato Healthy'])
print('Warnings:', report['warnings'])


## Train and evaluate
ImageNet transfer learning: three frozen-feature warm-up epochs followed by lower-rate full-network fine-tuning, maximum 30 epochs, early stopping after five non-improving validation epochs. Capped inverse-frequency class weights use training counts only; validation selection uses unweighted cross entropy. The test split is consumed only after best-validation checkpoint selection.

Outputs include actual accuracy, macro F1, all fourteen per-class precision/recall/support values, confusion matrix, individual test predictions, split provenance, versions, training history and the best checkpoint. There is no promised accuracy or runtime. If CUDA runs out of memory, restart training with a **new output directory** and `--batch-size 32`; retain the split.


In [ ]:
subprocess.run([sys.executable, '-m', 'training.multicrop.pipeline', 'train', '--epochs', '30', '--patience', '5', '--batch-size', '64'], check=True)


In [ ]:
evaluation = json.loads(Path('artifacts/phase4_run/evaluation.json').read_text())
print('Actual held-out accuracy:', evaluation['accuracy'])
print('Actual macro F1:', evaluation['f1_macro'])
for label, metrics in evaluation['per_class'].items():
    print(label, metrics)
from IPython.display import display, Image
display(Image(filename='artifacts/phase4_run/confusion_matrix.png'))


## Download the completed training run
The archive includes the checkpoint, evaluations, metadata and fourteen fixed held-out parity photos for local conversion. It is a private handoff archive, not a cleared public dataset release. No ONNX conversion or Android deployment is claimed by this notebook. Keep the original five-class files untouched.


In [ ]:
import shutil
archive = shutil.make_archive('/content/AgriVision-Phase4-trained-run', 'zip', root_dir='artifacts', base_dir='phase4_run')
files.download(archive)
print('Return this archive to artifacts/phase4_colab_return/ in the AgriVision project. Do not place it in models/.')
